# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\rm{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\rm{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\rm{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。

In [1]:
!pip install gensim
import gensim.downloader as api
import numpy as np

print("Google News Word2Vecモデルをダウンロード中...")
model = api.load("word2vec-google-news-300")
print("モデルのダウンロードと読み込みが完了")

Google News Word2Vecモデルをダウンロード中...
モデルのダウンロードと読み込みが完了


In [2]:
VOCAB_SIZE = len(model.key_to_index) + 1  # 300万単語 + <PAD>分
EMB_DIM = model.vector_size           # 300次元

#単語埋め込み行列
E = np.zeros((VOCAB_SIZE, EMB_DIM), dtype=np.float32)

word2id = {"<PAD>": 0}
id2word = {0: "<PAD>"}

#単語ベクトルを行列に格納し、辞書を構築
print("行列の構築中...")
for i, word in enumerate(model.index_to_key, start=1):
    E[i] = model[word]      # 行列のi行目にベクトルを格納
    word2id[word] = i      # 単語 -> ID
    id2word[i] = word      # ID -> 単語

#確認
print(f"行列の形状: {E.shape}")

行列の構築中...
行列の形状: (3000001, 300)


## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

In [10]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

import torch
import pandas as pd

def load_sst2_and_idize(filepath, word2id):
    df = pd.read_csv(filepath, sep='\t')
    dataset = []

    for text, label in zip(df['sentence'], df['label']):
        # 簡易的なトークナイズ（半角スペースで分割）
        tokens = text.split()
        # 語彙に含まれる単語のみをIDに変換
        input_ids = [word2id[word] for word in tokens if word in word2id]

        # 空のトークン列は除外
        if len(input_ids) > 0:
            dataset.append({
                'text': text,
                'label': torch.tensor([float(label)]),
                'input_ids': torch.tensor(input_ids)
            })
    return dataset

# データの読み込みと変換
train_data = load_sst2_and_idize('SST-2/train.tsv', word2id)
dev_data = load_sst2_and_idize('SST-2/dev.tsv', word2id)

# 確認
print(f"訓練データ数: {len(train_data)}")
print(f"開発データ数: {len(dev_data)}")
print("サンプル:", train_data[0])

--2026-05-29 03:44:16--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.238.192.60, 18.238.192.122, 18.238.192.99, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.238.192.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip.1’

SST-2.zip.1         100%[===================>]   7.09M  46.0MB/s    in 0.2s    

2026-05-29 03:44:17 (46.0 MB/s) - ‘SST-2.zip.1’ saved [7439277/7439277]

Archive:  SST-2.zip
replace SST-2/dev.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 訓練データ数: 66650
開発データ数: 872
サンプル: {'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])}


## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

In [11]:
import torch.nn as nn

class BoWModel(nn.Module):
    def __init__(self, weights_matrix, embed_dim):
        super().__init__()
        # 1. 事前学習済み埋め込みのロード（勾配更新なし）
        self.embedding = nn.Embedding.from_pretrained(torch.from_numpy(weights_matrix), freeze=True)
        # 2. 線形層 (300次元 -> 1次元)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, input_ids):
        # input_ids: [sequence_length]
        # 埋め込み取得: [sequence_length, embed_dim]
        embeds = self.embedding(input_ids)
        # 平均ベクトル算出: [embed_dim]
        feature_vec = torch.mean(embeds, dim=0)
        # 線形層 + シグモイド
        logits = self.fc(feature_vec)
        return torch.sigmoid(logits)

# モデルのインスタンス化
model_bow = BoWModel(E, EMB_DIM)
print(model_bow)

BoWModel(
  (embedding): Embedding(3000001, 300)
  (fc): Linear(in_features=300, out_features=1, bias=True)
)


## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

In [12]:
from torch.utils.data import DataLoader
import torch.optim as optim

# ハイパーパラメータの設定
LEARNING_RATE = 0.1
EPOCHS = 10

# 損失関数とオプティマイザ
criterion = nn.BCELoss()
optimizer = optim.SGD(model_bow.parameters(), lr=LEARNING_RATE)

# 学習ループ
print("学習を開始します...")
for epoch in range(EPOCHS):
    model_bow.train()
    total_loss = 0

    for item in train_data:
        input_ids = item['input_ids']
        label = item['label']

        # 勾配の初期化
        optimizer.zero_grad()

        # 順伝播
        output = model_bow(input_ids)

        # 損失の計算
        loss = criterion(output, label)

        # 逆伝播 + 重み更新
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_data)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

print("学習完了")

学習を開始します...
Epoch [1/10], Loss: 0.3884
Epoch [2/10], Loss: 0.3767
Epoch [3/10], Loss: 0.3763
Epoch [4/10], Loss: 0.3762
Epoch [5/10], Loss: 0.3762
Epoch [6/10], Loss: 0.3762
Epoch [7/10], Loss: 0.3762
Epoch [8/10], Loss: 0.3762
Epoch [9/10], Loss: 0.3762
Epoch [10/10], Loss: 0.3762
学習完了


## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

In [ ]:
def calculate_accuracy(model, dataset):
    model.eval()
    correct = 0
    with torch.no_grad():
        for item in dataset:
            input_ids = item['input_ids']
            label = item['label']

            # 予測値の計算
            output = model(input_ids)
            prediction = 1 if output >= 0.5 else 0

            if prediction == label.item():
                correct += 1

    return correct / len(dataset)

# 開発セットでの正解率を表示
accuracy = calculate_accuracy(model_bow, dev_data)
print(f"開発セットの正解率: {accuracy:.4f}")

開発セットの正解率: 0.7867


## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    # 1. 各事例を長さ（input_idsの要素数）の降順にソートする
    batch = sorted(batch, key=lambda x: len(x['input_ids']), reverse=True)

    # 2. データの取り出し
    texts = [item['text'] for item in batch]
    labels = [item['label'] for item in batch]
    input_ids_list = [item['input_ids'] for item in batch]

    # 3. パディング処理 (0で埋める)
    # pad_sequenceは [batch_size, max_len] の形状で返してくれる
    input_ids_padded = pad_sequence(input_ids_list, batch_first=True, padding_value=0)

    # 4. ラベルを1つのテンソルに結合
    labels_tensor = torch.stack(labels)

    return {
        'text': texts,
        'input_ids': input_ids_padded,
        'label': labels_tensor
    }

# 動作確認：訓練データの最初の4件でテスト
sample_batch = train_data[:4]
collated_sample = collate_fn(sample_batch)

print("--- Collate結果 ---")
print(f"input_ids shape: {collated_sample['input_ids'].shape}")
print(f"input_ids:\n{collated_sample['input_ids']}")
print(f"label:\n{collated_sample['label']}")

--- Collate結果 ---
input_ids shape: torch.Size([4, 11])
input_ids:
tensor([[     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,
           1276,   1964],
        [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,
              0,      0],
        [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,
              0,      0],
        [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,
              0,      0]])
label:
tensor([[1.],
        [0.],
        [0.],
        [0.]])


## 76. ミニバッチ学習


問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [ ]:
import time

# 1. DataLoaderの設定
# 訓練データをバッチサイズ64でシャッフルしてロード
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_fn)
# 評価データ
dev_loader = DataLoader(dev_data, batch_size=64, shuffle=False, collate_fn=collate_fn)

# 2. モデルの再初期化 (公平な比較のため)
model_batch = BoWModel(E, EMB_DIM)
optimizer = optim.SGD(model_batch.parameters(), lr=LEARNING_RATE)

# 3. ミニバッチ学習ループ
print("ミニバッチ学習を開始します...")
for epoch in range(EPOCHS):
    model_batch.train()
    total_loss = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch['input_ids']
        labels = batch['label']

        optimizer.zero_grad()

        # モデルのforwardをバッチ対応にする必要があります（現在は単一ベクトルの平均）
        # BoWModelの修正が必要なため、ここで修正版を再定義します
        outputs = torch.stack([model_batch.fc(torch.mean(model_batch.embedding(ids[ids != 0]), dim=0)) for ids in input_ids])
        outputs = torch.sigmoid(outputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}, Time: {time.time() - start_time:.2f}s")

# 4. 正解率の計算
def calculate_accuracy_batch(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids']
            labels = batch['label']

            # 簡易的なバッチ処理の適用
            outputs = torch.stack([torch.sigmoid(model.fc(torch.mean(model.embedding(ids[ids != 0]), dim=0))) for ids in input_ids])
            predictions = (outputs >= 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total

accuracy_batch = calculate_accuracy_batch(model_batch, dev_loader)
print(f"\n開発セットの正解率: {accuracy_batch:.4f}")

ミニバッチ学習を開始します...
Epoch [1/10], Loss: 0.5494, Time: 10.11s
Epoch [2/10], Loss: 0.4570, Time: 10.45s
Epoch [3/10], Loss: 0.4280, Time: 10.00s
Epoch [4/10], Loss: 0.4130, Time: 10.11s
Epoch [5/10], Loss: 0.4037, Time: 10.12s
Epoch [6/10], Loss: 0.3973, Time: 10.21s
Epoch [7/10], Loss: 0.3929, Time: 9.38s
Epoch [8/10], Loss: 0.3891, Time: 10.27s
Epoch [9/10], Loss: 0.3863, Time: 10.21s
Epoch [10/10], Loss: 0.3839, Time: 10.17s

開発セットの正解率: 0.7924


## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [ ]:
import torch

# 1. デバイスの設定 (GPUが利用可能ならcuda、そうでなければcpu)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

# 2. モデルをGPUへ転送
model_gpu = BoWModel(E, EMB_DIM).to(device)
optimizer = optim.SGD(model_gpu.parameters(), lr=LEARNING_RATE)

# 3. GPUを用いた訓練ループ
print("GPUでの学習を開始します...")
for epoch in range(EPOCHS):
    model_gpu.train()
    total_loss = 0
    start_time = time.time()

    for batch in train_loader:
        # データをGPUへ転送
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        # バッチ処理（GPU上）
        # ids != 0 のマスク処理を行って平均をとる
        outputs = []
        for ids in input_ids:
            valid_embeds = model_gpu.embedding(ids[ids != 0])
            mean_embed = torch.mean(valid_embeds, dim=0)
            outputs.append(model_gpu.fc(mean_embed))

        outputs = torch.sigmoid(torch.stack(outputs))

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}, Time: {time.time() - start_time:.2f}s")

# 4. 正解率の計算 (GPU版)
def calculate_accuracy_gpu(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            outputs = []
            for ids in input_ids:
                valid_embeds = model.embedding(ids[ids != 0])
                outputs.append(model.fc(torch.mean(valid_embeds, dim=0)))

            outputs = torch.sigmoid(torch.stack(outputs))
            predictions = (outputs >= 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total

accuracy_gpu = calculate_accuracy_gpu(model_gpu, dev_loader, device)
print(f"\n開発セットの正解率 (GPU): {accuracy_gpu:.4f}")

使用デバイス: cuda
GPUでの学習を開始します...
Epoch [1/10], Loss: 0.5499, Time: 24.19s
Epoch [2/10], Loss: 0.4572, Time: 23.08s
Epoch [3/10], Loss: 0.4281, Time: 23.60s
Epoch [4/10], Loss: 0.4131, Time: 22.82s
Epoch [5/10], Loss: 0.4038, Time: 22.62s
Epoch [6/10], Loss: 0.3974, Time: 22.13s
Epoch [7/10], Loss: 0.3927, Time: 22.36s
Epoch [8/10], Loss: 0.3891, Time: 22.48s
Epoch [9/10], Loss: 0.3863, Time: 22.67s
Epoch [10/10], Loss: 0.3840, Time: 22.57s

開発セットの正解率 (GPU): 0.7901


## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [13]:
import torch.nn as nn
import torch
import torch.optim as optim
import time
import gc # Import garbage collector
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

# 1. デバイスの設定 (GPUが利用可能ならcuda、そうでなければcpu)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

# ハイパーパラメータの設定 (このセルで定義)
LEARNING_RATE = 0.1
EPOCHS = 10

# 損失関数 (このセルで定義)
criterion = nn.BCELoss()

# collate_fnの再定義 (このセルでDataLoaderが利用できるように)
def collate_fn(batch):
    batch = sorted(batch, key=lambda x: len(x['input_ids']), reverse=True)
    texts = [item['text'] for item in batch]
    labels = [item['label'] for item in batch]
    input_ids_list = [item['input_ids'] for item in batch]
    input_ids_padded = pad_sequence(input_ids_list, batch_first=True, padding_value=0)
    labels_tensor = torch.stack(labels)
    return {
        'text': texts,
        'input_ids': input_ids_padded,
        'label': labels_tensor
    }

# DataLoaderの再定義 (このセルで利用できるように)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_data, batch_size=64, shuffle=False, collate_fn=collate_fn)

class BoWModelFineTune(nn.Module):
    def __init__(self, weights_matrix, embed_dim):
        super().__init__()
        # freeze=False にして、単語埋め込みも学習対象にする
        self.embedding = nn.Embedding.from_pretrained(torch.from_numpy(weights_matrix), freeze=False, sparse = True)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, input_ids):
        # input_ids: (batch_size, max_seq_len)
        # 1. パディング(0)ではない有効な単語の場所を記録したマスクを作成
        mask = (input_ids != 0).float() # (batch_size, max_seq_len)

        # 2. バッチ全体を一括でEmbeddingに通す
        embeds = self.embedding(input_ids) # (batch_size, max_seq_len, embed_dim)

        # 3. パディング部分のベクトルを0にする（マスクを掛ける）
        masked_embeds = embeds * mask.unsqueeze(-1) # (batch_size, max_seq_len, embed_dim)

        # 4. 有効な単語の数で平均をとる（平均プーリング）
        valid_counts = mask.sum(dim=1, keepdim=True).clamp(min=1.0) # (batch_size, 1)
        feature_vec = masked_embeds.sum(dim=1) / valid_counts # (batch_size, embed_dim)

        # 5. 全結合層とSigmoidを一括適用
        logits = self.fc(feature_vec) # (batch_size, 1)
        return torch.sigmoid(logits)

# --- OOMエラー対策: 以前のモデルをGPUメモリから解放 ---
if 'model_gpu' in globals() and isinstance(model_gpu, nn.Module):
    print("Attempting to clear previous model from GPU memory...")
    del model_gpu
    # Set to None explicitly to help GC
    model_gpu = None
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("Previous model (model_gpu) and CUDA cache cleared.")
elif 'model_gpu' in globals() and model_gpu is None:
    print("model_gpu was already None.")
else:
    print("No previous model_gpu found or it was not an nn.Module.")

# モデルの初期化とGPU転送
try:
    model_ft = BoWModelFineTune(E, EMB_DIM).to(device)
    optimizer = optim.SGD(model_ft.parameters(), lr=LEARNING_RATE)

    print("ファインチューニングありの学習を開始します...")
    for epoch in range(EPOCHS):
        model_ft.train()
        total_loss = 0
        start_time = time.time()

        for batch in train_loader:
                input_ids = batch['input_ids'].to(device) # shape: (batch_size, max_seq_len)
                labels = batch['label'].to(device)        # shape: (batch_size, 1)

                optimizer.zero_grad()

                # モデルのforwardメソッドでバッチ処理を一括実行
                outputs = model_ft(input_ids)

                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}, Time: {time.time() - start_time:.2f}s")

    # calculate_accuracy_gpu関数もモデルのforwardを呼び出すように修正
    def calculate_accuracy_gpu(model, loader, device):
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in loader:
                input_ids = batch['input_ids'].to(device)
                labels = batch['label'].to(device)

                outputs = model(input_ids) # モデルのforwardメソッドでバッチ処理を一括実行

                predictions = (outputs >= 0.5).float()
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        return correct / total

    accuracy_ft = calculate_accuracy_gpu(model_ft, dev_loader, device)
    print(f"\n開発セットの正解率 (Fine-tuning): {accuracy_ft:.4f}")

except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print(f"\nGPUメモリ不足のため、学習を開始できませんでした: {e}")
        print("この問題が続く場合は、Colabランタイムを再起動してみてください (ランタイム -> ランタイムを再起動)。")
    else:
        raise e

使用デバイス: cuda
No previous model_gpu found or it was not an nn.Module.
ファインチューニングありの学習を開始します...
Epoch [1/10], Loss: 0.5452, Time: 2.50s
Epoch [2/10], Loss: 0.4375, Time: 1.96s
Epoch [3/10], Loss: 0.3951, Time: 1.96s
Epoch [4/10], Loss: 0.3697, Time: 2.33s
Epoch [5/10], Loss: 0.3516, Time: 1.92s
Epoch [6/10], Loss: 0.3376, Time: 1.95s
Epoch [7/10], Loss: 0.3259, Time: 1.94s
Epoch [8/10], Loss: 0.3157, Time: 1.91s
Epoch [9/10], Loss: 0.3067, Time: 1.92s
Epoch [10/10], Loss: 0.2986, Time: 2.36s

開発セットの正解率 (Fine-tuning): 0.8188


## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。

### 79. アーキテクチャの変更 (多層パーセプトロンの実装)

隠れ層を1つ追加し、ドロップアウト（過学習抑制）を導入した多層パーセプトロン (MLP) を実装。

In [14]:
import torch.nn.functional as F

class MLPModel(nn.Module):
    def __init__(self, weights_matrix, embed_dim, hidden_dim=256):
        super().__init__()
        # 1. 埋め込み層 (ファインチューニングあり)
        self.embedding = nn.Embedding.from_pretrained(torch.from_numpy(weights_matrix), freeze=False, sparse=True)

        # 2. 隠れ層
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3) # ドロップアウトで過学習を防止

        # 3. 出力層
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        # input_ids: (batch_size, max_seq_len)
        mask = (input_ids != 0).float()
        embeds = self.embedding(input_ids)

        # 平均プーリング
        valid_counts = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        feature_vec = (embeds * mask.unsqueeze(-1)).sum(dim=1) / valid_counts

        # MLP部分
        x = self.fc1(feature_vec)
        x = F.relu(x)        # 活性化関数
        x = self.dropout(x)  # ドロップアウト
        logits = self.fc2(x)
        return torch.sigmoid(logits)

# モデルの初期化
hidden_dim = 256
model_mlp = MLPModel(E, EMB_DIM, hidden_dim).to(device)
optimizer = optim.SGD(model_mlp.parameters(), lr=LEARNING_RATE)

print("MLPモデルでの学習を開始します...")
for epoch in range(EPOCHS):
    model_mlp.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model_mlp(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

accuracy_mlp = calculate_accuracy_gpu(model_mlp, dev_loader, device)
print(f"\n開発セットの正解率 (MLP): {accuracy_mlp:.4f}")

MLPモデルでの学習を開始します...
Epoch [1/10], Loss: 0.4831
Epoch [2/10], Loss: 0.3438
Epoch [3/10], Loss: 0.3129
Epoch [4/10], Loss: 0.2916
Epoch [5/10], Loss: 0.2730
Epoch [6/10], Loss: 0.2587
Epoch [7/10], Loss: 0.2470
Epoch [8/10], Loss: 0.2360
Epoch [9/10], Loss: 0.2238
Epoch [10/10], Loss: 0.2165

開発セットの正解率 (MLP): 0.8268
